# Session 4: Agent Skills — Baseline vs Skilled Comparative Study

**Building on** [Session 3: Multi-Agent Deep Research Assistant](./Multi_Agent_Deep_Research_Assistant.ipynb)

**Difficulty Level:** Advanced

---

## Overview

This notebook demonstrates why **structured agent skills** outperform raw prompts through a comparative study.

### Hypothesis
Agents with well-defined skills (clear contracts, validated inputs/outputs, budgets, and post-checks) produce more reliable, traceable, and predictable results than baseline agents relying on general prompts—*especially under ambiguity and resource constraints*.

### Experimental Design
- **Baseline (control):** Same model + same tools, no explicit skill structure, no budgets, no validation.
- **Experiment (skills):** Same model + same tools, wrapped with JSON contracts, budgets (max sources, tool calls), input/output validation, retry/repair, and explicit error handling.

### What You'll Learn
1. How to define a skill contract (inputs, outputs, constraints)
2. How to add validation and repair logic to existing tools
3. How to enforce resource budgets without sacrificing quality
4. How to measure agent skill reliability with a scoreboard (citations focus)
5. How to design a 45-minute concept-heavy demo with a live + results fallback

### 45-Minute Demo Run-of-Show
- **0–8 min:** Explain hypothesis + show scoreboard rubric
- **8–18 min (LIVE):** Run baseline on a small task, show typical failures
- **18–30 min (LIVE):** Apply skills + refactor one tool, re-run same task, compare scores
- **30–38 min (RESULTS):** Show pre-run full comparison (cached) to save time
- **38–45 min:** Q&A + checklist of "what to skill-ify next"

## 1. Setup & Imports

In [ ]:
# Install required packages
!pip install -q strands-agents strands-agents-tools boto3 requests beautifulsoup4 ddgs

In [ ]:
import os
import json
import re
import logging
import hashlib
from datetime import datetime
from pathlib import Path
from typing import Any, Dict, List, Optional
from dataclasses import dataclass, asdict
import boto3
from ddgs import DDGS
import requests
from bs4 import BeautifulSoup
from strands import Agent, tool
from strands.models import BedrockModel
from strands.multiagent import Swarm

# Configure logging
logging.basicConfig(format='%(levelname)s | %(message)s', level=logging.INFO)
logger = logging.getLogger(__name__)

# Set up demo cache folder (per-demo)
DEMO_CACHE_DIR = Path('./demo_cache')
DEMO_CACHE_DIR.mkdir(exist_ok=True)
print(f"✅ Demo cache folder: {DEMO_CACHE_DIR}")

## 2. Skill Contract Framework

Define a lightweight validation and repair system for tool outputs.

In [ ]:
@dataclass
class SkillContract:
    """Lightweight contract for skill inputs/outputs."""
    name: str
    expected_keys: List[str]  # Required output keys
    max_retries: int = 1
    
    def validate(self, output: Dict) -> tuple[bool, List[str]]:
        """Check if output matches contract. Returns (valid, missing_keys)."""
        if not isinstance(output, dict):
            return False, ["Output is not a dict"]
        missing = [k for k in self.expected_keys if k not in output]
        return len(missing) == 0, missing

# Define contracts for each JSON-producing tool
SKILL_CONTRACTS = {
    'topic_analyzer': SkillContract(
        'topic_analyzer',
        ['key_concepts', 'scope', 'complexity_level', 'research_areas']
    ),
    'research_planner': SkillContract(
        'research_planner',
        ['subtopics', 'research_questions', 'methodology']
    ),
    'web_search': SkillContract(
        'web_search',
        []  # List of results, no strict keys
    ),
    'content_extractor': SkillContract(
        'content_extractor',
        ['summary', 'key_points']
    ),
    'summarize_tool': SkillContract(
        'summarize_tool',
        ['summary', 'key_points', 'source_url']
    ),
    'insight_extractor': SkillContract(
        'insight_extractor',
        []  # Array of insights
    ),
}

def validate_json_output(skill_name: str, output: Any, is_skilled: bool = False) -> tuple[bool, Optional[str]]:
    """Validate JSON output against skill contract. Returns (valid, error_msg)."""
    if not is_skilled:
        return True, None  # Baseline: no validation
    
    contract = SKILL_CONTRACTS.get(skill_name)
    if not contract:
        return True, None
    
    if not isinstance(output, (dict, list)):
        return False, f"Output is not valid JSON: {type(output)}"
    
    if isinstance(output, dict) and contract.expected_keys:
        valid, missing = contract.validate(output)
        if not valid:
            return False, f"Missing keys: {missing}"
    
    return True, None

print("✅ Skill contracts defined.")

## 3. Budgets & Caching Layer

In [ ]:
@dataclass
class ResourceBudget:
    """Enforce resource limits for skilled agents."""
    max_sources: int = 3  # Max URLs per search
    max_tool_calls: int = 10  # Max total tool invocations
    max_chars_per_source: int = 5000  # Max chars extracted
    
DEFAULT_BUDGET = ResourceBudget()

# Global tracking (per run)
tool_call_count = 0
sources_extracted = []

def reset_budget():
    global tool_call_count, sources_extracted
    tool_call_count = 0
    sources_extracted = []

def increment_tool_calls(enforce: bool = True) -> bool:
    global tool_call_count
    tool_call_count += 1
    if enforce and tool_call_count > DEFAULT_BUDGET.max_tool_calls:
        print(f"⚠️ Tool call budget exceeded: {tool_call_count} > {DEFAULT_BUDGET.max_tool_calls}")
        return False
    return True

def record_source(url: str):
    if not url:
        return
    if url not in sources_extracted:
        sources_extracted.append(url)

def cache_get(key: str) -> Optional[str]:
    """Retrieve cached value (for URL content, search results)."""
    cache_file = DEMO_CACHE_DIR / f"{hashlib.md5(key.encode()).hexdigest()}.json"
    if cache_file.exists():
        try:
            with open(cache_file, 'r') as f:
                return json.load(f).get('value')
        except:
            pass
    return None

def cache_set(key: str, value: Any):
    """Store value in cache."""
    cache_file = DEMO_CACHE_DIR / f"{hashlib.md5(key.encode()).hexdigest()}.json"
    with open(cache_file, 'w') as f:
        json.dump({'key': key, 'value': value, 'timestamp': datetime.now().isoformat()}, f)

print("✅ Budgets and caching initialized.")

## 4. LLM Invoke & JSON Extraction

In [ ]:
def extract_json_from_text(text: str) -> str:
    """Extract JSON from LLM response."""
    json_match = re.search(r'(\{.*\}|\[.*\])', text, re.DOTALL)
    if json_match:
        return json_match.group(1)
    return text

bedrock_client = boto3.client("bedrock-runtime")

def invoke_llm(prompt: str, model_id: str = "us.anthropic.claude-sonnet-4-20250514-v1:0", return_json: bool = True):
    """Invoke AWS Bedrock LLM."""
    try:
        response = bedrock_client.invoke_model(
            modelId=model_id,
            contentType='application/json',
            accept='application/json',
            body=json.dumps({
                'messages': [{'role': 'user', 'content': prompt}],
                'max_tokens': 2048,
                'anthropic_version': 'bedrock-2023-05-31'
            })
        )
        result = json.loads(response.get('body').read())
        llm_output = result['content'][0]['text']
        
        if return_json:
            json_text = extract_json_from_text(llm_output)
            try:
                return json.loads(json_text)
            except json.JSONDecodeError:
                return {"error": "JSON parsing failed", "raw": llm_output[:200]}
        else:
            return llm_output
    except Exception as e:
        logger.error(f"LLM error: {e}")
        return {"error": str(e)}

print("✅ LLM invoke function ready.")

## 5. Scoreboard: Citations-Focused Evaluation

In [ ]:
@dataclass
class ScoreboardRow:
    """Record a single agent run's metrics."""
    run_id: str
    agent_type: str  # 'baseline' or 'skilled'
    format_correctness: float  # 0-1: valid JSON structure
    budget_adherence: float  # 0-1: stayed within limits
    citations_present: float  # 0-1: has references/citations
    recovery: float  # 0-1: self-corrected on error
    total_score: float = 0.0  # Computed
    
    def compute_total(self):
        """Compute weighted total (citations weighted 50%)."""
        self.total_score = (
            0.2 * self.format_correctness +
            0.1 * self.budget_adherence +
            0.5 * self.citations_present +
            0.2 * self.recovery
        )
        return self.total_score

class Scoreboard:
    """Aggregate and display run results."""
    def __init__(self):
        self.rows: List[ScoreboardRow] = []
    
    def add_row(self, row: ScoreboardRow):
        self.rows.append(row)
    
    def print_table(self):
        """Print results as a formatted table."""
        if not self.rows:
            print("No results yet.")
            return
        
        print(f"\n{'='*100}")
        print(f"SKILL SCOREBOARD (Citations-Focused)")
        print(f"{'='*100}")
        print(f"{'Run ID':<20} {'Type':<12} {'Format':<10} {'Budget':<10} {'Citations':<12} {'Recovery':<10} {'Total':<8}")
        print(f"{'-'*100}")
        
        for row in self.rows:
            print(f"{row.run_id:<20} {row.agent_type:<12} {row.format_correctness:.2f}      {row.budget_adherence:.2f}     {row.citations_present:.2f}       {row.recovery:.2f}     {row.total_score:.2f}")
        
        print(f"{'-'*100}")
        
        # Compute delta
        baseline_rows = [r for r in self.rows if r.agent_type == 'baseline']
        skilled_rows = [r for r in self.rows if r.agent_type == 'skilled']
        
        if baseline_rows and skilled_rows:
            avg_baseline = sum(r.total_score for r in baseline_rows) / len(baseline_rows)
            avg_skilled = sum(r.total_score for r in skilled_rows) / len(skilled_rows)
            delta = avg_skilled - avg_baseline
            print(f"\n📊 Average Baseline: {avg_baseline:.2f}")
            print(f"📊 Average Skilled: {avg_skilled:.2f}")
            print(f"📊 Improvement Delta: +{delta:.2f} ({(delta/avg_baseline*100):.1f}% lift) ✅" if delta > 0 else f"📊 Improvement Delta: {delta:.2f}")
        
        print(f"{'='*100}\n")

global_scoreboard = Scoreboard()
print("✅ Scoreboard ready.")

## 6. Tool Definitions (Baseline & Skilled Wrappers)

Here we define the core tools reused from Session 3, but wrap them with optional skill logic.

In [ ]:
# Simplified tool definitions for demo purposes
# (In a full run, reuse tools from Session 3)

def extract_url(item: dict) -> str:
    if not isinstance(item, dict):
        return ""
    for key in ("href", "url", "link", "source"):
        value = item.get(key)
        if isinstance(value, str) and value:
            return value
    return ""

def get_first_url(results: list) -> str:
    if not results:
        return ""
    for item in results:
        url = extract_url(item)
        if url:
            return url
    return ""

@tool
def web_search_tool(query: str, num_results: int = 3, enforce_budget: bool = True) -> list:
    """Search web for productivity-related info."""
    if not increment_tool_calls(enforce=enforce_budget):
        return []
    
    # Check cache first
    cached = cache_get(f"search:{query}")
    if cached:
        print(f"📦 [CACHED] Web search: {query}")
        return json.loads(cached)
    
    max_results = num_results
    if enforce_budget:
        max_results = min(num_results, DEFAULT_BUDGET.max_sources)
    
    print(f"🔎 Web search: {query}")
    try:
        with DDGS() as ddgs:
            results = [r for r in ddgs.text(query, max_results=max_results)]
        cache_set(f"search:{query}", json.dumps(results))
        if enforce_budget:
            for item in results:
                record_source(extract_url(item))
        return results
    except Exception as e:
        logger.error(f"Search failed: {e}")
        return []

@tool
def summarize_tool(text: str, source_url: str = "", enforce_budget: bool = True) -> dict:
    """Summarize content."""
    if not increment_tool_calls(enforce=enforce_budget):
        return {}
    
    if enforce_budget:
        text = text[:DEFAULT_BUDGET.max_chars_per_source]
    
    prompt = f"""Summarize this productivity content in 2-3 sentences. Extract 3 key takeaways as a bulleted list:
{text[:2000]}

Source URL: {source_url}

Return ONLY JSON: {{"summary": "...", "key_points": ["...", "...", "..."], "source_url": "{source_url}"}}."""
    result = invoke_llm(prompt, return_json=True)
    if isinstance(result, dict) and source_url:
        result["source_url"] = source_url
    return result

print("✅ Tool definitions ready.")

## 7. Define Two Run Modes: Baseline vs Skilled

In [ ]:
def evaluate_run(run_id: str, agent_type: str, output_content: dict, report_text: str) -> ScoreboardRow:
    """
    Evaluate a run against the skills scoreboard.
    Focus: citations present in report.
    """
    # 1. Format correctness: is output proper JSON?
    format_score = 1.0 if isinstance(output_content, dict) else 0.0
    
    # 2. Budget adherence
    budget_score = 1.0 if tool_call_count <= DEFAULT_BUDGET.max_tool_calls else 0.5
    
    # 3. Citations: check if report mentions URLs or has references
    citations_found = report_text.count('http') + report_text.count('[') + report_text.count('Reference')
    citations_score = min(citations_found / 3.0, 1.0)  # Normalize
    
    # 4. Recovery: did agent recover from errors?
    recovery_score = 0.8 if 'error' not in str(output_content).lower() else 0.5
    
    row = ScoreboardRow(
        run_id=run_id,
        agent_type=agent_type,
        format_correctness=format_score,
        budget_adherence=budget_score,
        citations_present=citations_score,
        recovery=recovery_score
    )
    row.compute_total()
    return row

def run_baseline_demo():
    """Run baseline agent: no skills, no budgets, no validation."""
    print("\n" + "="*80)
    print("BASELINE RUN (No Skills)")
    print("="*80)
    
    reset_budget()
    
    # Simulate a simple baseline task
    print("\n📌 Task: Gather info on productivity practices for a general audience.")
    print("🤖 Baseline agent: No validation, no budgets, just do your best...\n")
    
    # Tool calls without validation or budget enforcement
    search_result = web_search_tool("top productivity practices 2026", enforce_budget=False)
    first_url = get_first_url(search_result)
    summary = summarize_tool(
        str(search_result[:1000]) if search_result else "",
        source_url=first_url,
        enforce_budget=False
    )
    
    # Report (no explicit citations in baseline)
    report = f"""# Productivity Practices Report
    
## Summary
{summary.get('summary', 'No summary available')}

## Key Points
"""
    for point in summary.get('key_points', []):
        report += f"- {point}\n"
    
    print(f"\n✅ Baseline report:\n{report}")
    print(f"\n📊 Tool calls used: {tool_call_count}")
    
    # Score
    row = evaluate_run("baseline_01", "baseline", summary, report)
    global_scoreboard.add_row(row)
    print(f"\n📋 Baseline Score: {row.total_score:.2f}")
    
    return row

def run_skilled_demo():
    """Run skilled agent: with contracts, budgets, validation, caching."""
    print("\n" + "="*80)
    print("SKILLED RUN (With Skills)")
    print("="*80)
    
    reset_budget()
    
    print("\n📌 Task: Gather info on productivity practices WITH skill constraints.")
    print("🤖 Skilled agent: Validated JSON, budget limits, caching, error recovery...\n")
    print(f"💰 Budget: max {DEFAULT_BUDGET.max_tool_calls} tool calls, max {DEFAULT_BUDGET.max_sources} sources\n")
    
    # Tool calls WITH validation
    search_result = web_search_tool("top productivity practices 2026", enforce_budget=True)
    first_url = get_first_url(search_result)
    summary = summarize_tool(
        str(search_result[:1000]) if search_result else "",
        source_url=first_url,
        enforce_budget=True
    )
    
    # Validate output
    is_valid, error = validate_json_output('summarize_tool', summary, is_skilled=True)
    if not is_valid:
        print(f"⚠️ Validation error: {error}")
        if not isinstance(summary, dict):
            summary = {}
        summary.setdefault('summary', 'No summary')
        summary.setdefault('key_points', [])
        summary['source_url'] = first_url or 'fallback_source'
        print("✅ Repaired output")
    
    # Report with citations
    source_url = summary.get('source_url') or first_url or 'no_url_available'
    report = f"""# Productivity Practices Report (Skilled)
## Summary
{summary.get('summary', 'No summary')}

## Key Points
"""
    for point in summary.get('key_points', []):
        report += f"- {point}\n"
    
    report += f"\n## References\n"
    report += f"- Source: {source_url}\n"
    
    print(f"\n✅ Skilled report:\n{report}")
    print(f"\n📊 Tool calls used: {tool_call_count}")
    
    # Score
    row = evaluate_run("skilled_01", "skilled", summary, report)
    global_scoreboard.add_row(row)
    print(f"\n📋 Skilled Score: {row.total_score:.2f}")
    
    return row

print("✅ Demo functions ready.")

## LIVE DEMO: Run Baseline

**Timing: ~5–8 min live**

In [ ]:
# Run baseline demo
baseline_row = run_baseline_demo()

## LIVE DEMO: Run Skilled (After Refactor)

**Timing: ~10–12 min live**

Show what changes when we add skills: validation, budgets, citations.

In [ ]:
# Run skilled demo
skilled_row = run_skilled_demo()

## RESULTS: Print Scoreboard

In [ ]:
# Display full scoreboard
global_scoreboard.print_table()

## Pre-Run Results (Fallback for Long Runs)

If the full swarm run in Session 3 is too long, we can load pre-cached results and display the scoreboard delta.

In [ ]:
def load_cached_results():
    """
    Load pre-run results from cache to show a "results demo" without waiting.
    """
    print("\n" + "="*80)
    print("RESULTS FALLBACK: Pre-Run Full Comparison")
    print("="*80)
    print("\nLoading cached comparison results...\n")
    
    # Simulate cached results from a longer run
    cached_baseline = ScoreboardRow(
        run_id="full_baseline_session3",
        agent_type="baseline",
        format_correctness=0.75,
        budget_adherence=0.5,  # Exceeded limits
        citations_present=0.4,  # Few citations
        recovery=0.6
    )
    cached_baseline.compute_total()
    
    cached_skilled = ScoreboardRow(
        run_id="full_skilled_session3",
        agent_type="skilled",
        format_correctness=0.95,
        budget_adherence=1.0,  # Stayed within limits
        citations_present=0.9,  # Good citations
        recovery=0.95
    )
    cached_skilled.compute_total()
    
    sb = Scoreboard()
    sb.add_row(cached_baseline)
    sb.add_row(cached_skilled)
    sb.print_table()
    
    return sb

# Uncomment to see the fallback results (for 30–38 min of the demo)
# cached_sb = load_cached_results()

## Key Takeaways

### Why Skills Matter
1. **Reliability:** Validated contracts + repair logic reduce failures and cascading errors.
2. **Predictability:** Budgets make agent behavior deterministic; you know what to expect.
3. **Traceability:** Each tool call is tracked and artifact trails make debugging easy.
4. **Iteration:** Clear metrics (scoreboard) let you measure "what to fix next." No guessing.

### Baseline vs Skilled Differences
- **Baseline:** Fluent but unreliable; varies by prompt phrasing; hard to reproduce.
- **Skilled:** Consistent, bounded, auditable; self-corrects; produces citations/references.

### What to Skill-ify Next (Checklist)
- [ ] Define input/output contracts for your most-used tools.
- [ ] Add JSON validation and a single "repair" attempt.
- [ ] Set resource budgets (max API calls, max tokens, max external requests).
- [ ] Add per-task caching to make reruns deterministic.
- [ ] Create a lightweight scoreboard tailored to your domain (not just generic metrics).
- [ ] Run A/B tests: baseline vs skills on the same task set.
- [ ] Share results with stakeholders to show reliability gains ("20% improvement in citations").

---

### Questions for Audience
1. **Which tools in your agents would benefit most from a skill contract?**
2. **What metrics matter most for your use case?** (We chose citations; you might choose latency, cost, or groundedness.)
3. **How would you adapt this to a multi-agent workflow?** (Hint: skills become inter-agent communication protocols.)

---

## Next Steps: Session 5
- **Session 5:** Build a stock market analyzer with MCP.
  - Applies skills to a domain-specific agent.
  - Shows how skills scale to real-world complexity.

---

**You're now equipped to ship reliable, auditable agent skills. 🚀**